**Quest 16: Cat Grin of Fortune**

![Header image](./imgs/image_quest_16.svg)
[<img src="./imgs/instagram.svg" alt="My SVG" width="15" height="15"><small>_Monika Lipińska_</small>](https://www.instagram.com/monli_art/)

---

## Part I

> ### :book: Story section
>
> You find yourself strolling through the streets of the town buzzing with excitement for the Tournament. As you meander along the cobblestone paths, you can't help but notice the Byter Paelish, standing proudly in front of his Entertainment Hub with a peculiar contraption by his side that looks like a mix between a jukebox and a magical slot machine.
> <br/><br/>

The machine, or as the Byter put it, the "Cat Grin of Fortune" features a window showcasing a line-up of whimsical cat faces: left eye, muzzle, and right eye, like a happy cat represented by `^_^`. A hefty lever on the right side seems to control the fate of these cat codes.

The Byter pulls the right lever and cat faces scroll in the window, appearing to be engraved on wheels . This explains the machine's size compared to the window. After a suspenseful moment, the wheels halt one after another, revealing a new sequence of cats.

To the tune of lute strings and catchy lyrics, Paelish spills the beans on the rules of the game. For every trio of identical symbols in the whimsically random cat sequence, you'll be rewarded with one Byte Coin. But the fun doesn't stop there - every additional symbol matching the trio adds another coin to your prize!

The Byter explains there are no secrets in the Cat Grin of Fortune. The side of the machine has the operating instructions and the wheel configurations <span style="color:Yellow">( your notes )</span> engraved on it. Above the image screen, a counter displays the number of right lever pulls since the machine started operating.

As you delve into the instructions, it becomes apparent that this contraption isn't just a random assortment of items. It is a simple yet whimsical masterpiece, where <span style="color:Ivory">ach wheel spins a sequence of cat faces</span>, represented for simplicity as a vertical strip. Before the right lever is pulled for the first time, the wheels are set to display the first symbol of each strip. <span style="color:Ivory">The numbers above the strips show how many positions each wheel turns with a single pull of the right lever</span>. You wonder if it is easy to predict the next sequence on the wheels. The counter currently shows number 99, so you need to predict the 100th sequence.

<span style="color:Ivory">**Example based on the following notes:**<span>

<pre style="border: dashed green;width:fit-content;border-radius:5px;padding:2px;">
1,2,3

^_^ -.- ^,-
>.- ^_^ >.<
-_- -.- >.<
    -.^ ^_^
    >.>
</pre>

The first line contains the number of <span style="color:Ivory">positions each wheel turns with a single pull of the right lever</span>. The rest of the input represents the sequence of symbols on each wheel as vertical strips. The machine starts by displaying the first trio of symbols:` ^_^ -.- ^,-`.

After the first pull, the first wheel turns by 1, the second by 2, and the third by 3 positions, resulting in the new sequence: `>.- -.- ^_^` which is worth <span style="color:Ivory">1 Byte Coin</span> for the `-` triple.

After the second pull, the wheels turn again in the same way, resulting in: `-_- >.> >.<` which is also worth <span style="color:Ivory">1 Byte Coin</span> for the '>' triple. Below you can see the results of pulling the right lever several pulls, followed by the number of coins won.

<pre style="border: dashed green;width:fit-content;border-radius:5px;padding:2px;">
Pull       Result      Byte Coins Won
  0:    ^_^ -.- ^,-          -
  1:    >.- -.- ^_^          1
  2:    -_- >.> >.<          1
  3:    ^_^ ^_^ >.<          2   (one extra for 4th ^ symbol)
  4:    >.- -.^ ^,-          1
  5:    -_- -.- ^_^          2   (one extra for 4th - symbol)
...
 21:    ^_^ -.- ^_^          2   (one extra for 4th ^ symbol)
...
 33:    ^_^ ^_^ ^_^          5   (one coin for _ trio plus 4 coins for six ^ symbols) 
...
100:    >.- -.- ^,-          2
...
</pre>

For this example, the 100th sequence of the Cat Grin of Fortune is `>.- -.- ^,-`.

<span style="color:Yellow">What is the 100th sequence produced by the Byter's machine?</span>


In [13]:
from collections import Counter
from itertools import batched, chain, zip_longest
from math import lcm

from cycler import V
from numpy.char import splitlines

from util import Str
import re
from test_utilities import test

notes = """
    1,2,3

    ^_^ -.- ^,-
    >.- ^_^ >.<
    -_- -.- >.<
        -.^ ^_^
        >.>
"""

tests = [
    {
        "name": "Example Part I 100 pulls",
        "notes": notes,
        "pulls": 100,
        "expected": ">.- -.- ^,-",
    },
    {
        "name": "Example Part I 0 pulls",
        "notes": notes,
        "pulls": 0,
        "expected": "^_^ -.- ^,-",
    },
    {
        "name": "Example Part I 1 pull",
        "notes": notes,
        "pulls": 1,
        "expected": ">.- -.- ^_^",
    },
    {
        "name": "Example Part I 2 pulls",
        "notes": notes,
        "pulls": 2,
        "expected": "-_- >.> >.<",
    },
    {
        "name": "Example Part I pulls 3",
        "notes": notes,
        "pulls": 3,
        "expected": "^_^ ^_^ >.<",
    },
    {
        "name": "Example Part I 21 pulls",
        "notes": notes,
        "pulls": 21,
        "expected": "^_^ -.- ^_^",
    },
]


class Byter(Str):
    def __init__(self, notes: str) -> None:
        deltas, wheels = self._parse(notes)

        self.deltas = deltas
        self.wheels = wheels

    def pull_lever_pulls(self, pulls: int = 100) -> str:
        indici = [0] * len(self.deltas)

        for i in range(len(indici)):
            indici[i] = pulls * self.deltas[i] % len(self.wheels[i])

        return self.to_str(indici)

    def to_str(self, indici):
        return " ".join(self.wheels[i][indici[i]] for i in range(len(self.deltas)))

    def _parse(self, notes):
        deltas, wheels = re.split(r"\n\s*\n+", notes.strip())

        deltas = [int(d) for d in deltas.split(",")]

        prefix = wheels[: re.search(r"\S", wheels).start()]  # type: ignore
        wheels = [
            [
                "".join(c for c in t if c != " ")
                for t in batched(w.removeprefix(prefix), 4)
            ]
            for w in wheels.splitlines()
        ]
        wheels = [[b for b in w if b] for w in zip_longest(*wheels)]
        return deltas, wheels


@test(tests=tests[:])
def part_I(notes: str, pulls: int = 100) -> str:
    byter = Byter(notes)
    return byter.pull_lever_pulls(pulls)


Test Example Part I 100 pulls passed, for part_I.
Test Example Part I 0 pulls passed, for part_I.
Test Example Part I 1 pull passed, for part_I.
Test Example Part I 2 pulls passed, for part_I.
Test Example Part I pulls 3 passed, for part_I.
Test Example Part I 21 pulls passed, for part_I.
Success


In [14]:
with open("../inputs/everybody_codes_e2024_q16_p1.txt") as f:
    notes1 = f.read()

print(f"Part I: {part_I(notes1)}")

Part I: -_^ ^_^ >.- >.-


## Part II

As soon as the next sequence appears in the machine's window, you hear horns announcing a gathering of tournament participants at Paelish's estate. The Knights of the Order decide to check the fairness of the owner's machines and at the same time play the next round of the tournament.

The knights enter the Entertainment Hub, where there are dozens of machines similar to the one at the entrance, but their wheel schemes are much more complex. Additionally, the game instructions are slightly different. The muzzles of the cats are ignored in the search for matching symbols. Only the eyes are interpreted, which likely reduces the chances of winning by a significant amount.
Each knight's task is to calculate the number of coins won so far on the machine they are analysing. You approach your machine <span style="color:yellow">( your notes )</span>. The right lever pull counter shows the value <pre style="color: ivory;border: 1px dashed lightyellow;width:fit-content; padding:5px;border-radius: 5px;">202420242024</pre>.

Example based on the following notes:

<pre style="border: dashed green;width:fit-content;border-radius:5px;padding:2px;">
1,2,3

^_^ -.- ^,-
>.- ^_^ >.<
-_- -.- >.<
    -.^ ^_^
    >.>
</pre>

For this example, the total number of Byte Coins won after pulling the right lever several pulls is as follows:

<pre style="border: dashed green;width:fit-content;border-radius:5px;padding:2px;">
           Pull        Total Byte Coins 
             0:                       -
             1:                       1
             2:                       2
             3:                       4
             4:                       5
             5:                       7
            ...                     ...
            10:                      15
           100:                     138
          1000:                    1383
         10000:                   13833
        100000:                  138333
       1000000:                 1383333
      10000000:                13833333
     100000000:               138333333
    1000000000:              1383333333
   10000000000:             13833333333
  100000000000:            138333333333
  202420242024:            280014668134
</pre>

<span style="color:Yellow">What is the total number of Byte Coins won so far on the machine you are verifying after `202420242024` pulls of the right lever?</span>


In [15]:
from test_utilities import test
from collections import Counter
from itertools import chain

notes = """
1,2,3

^_^ -.- ^,-
>.- ^_^ >.<
-_- -.- >.<
    -.^ ^_^
    >.>
"""

tests = [
    {
        "name": "Example Part II 202420242024 pulls",
        "notes": notes,
        "pulls": 202420242024,
        "expected": 280014668134,
    },
    {
        "name": "Example Part II 0 pull",
        "notes": notes,
        "pulls": 0,
        "expected": 0,
    },
    {
        "name": "Example Part II 1 time",
        "notes": notes,
        "pulls": 1,
        "expected": 1,
    },
    {
        "name": "Example Part II 2 pulls",
        "notes": notes,
        "pulls": 2,
        "expected": 2,
    },
    {
        "name": "Example Part II 3 pulls",
        "notes": notes,
        "pulls": 3,
        "expected": 4,
    },
    {
        "name": "Example Part II 4 pulls",
        "notes": notes,
        "pulls": 4,
        "expected": 5,
    },
    {
        "name": "Example Part II 5 pulls",
        "notes": notes,
        "pulls": 5,
        "expected": 7,
    },
    {
        "name": "Example Part II 10 pulls",
        "notes": notes,
        "pulls": 10,
        "expected": 15,
    },
    {
        "name": "Example Part II 100 pulls",
        "notes": notes,
        "pulls": 100,
        "expected": 138,
    },
    {
        "name": "Example Part II 1000 pulls",
        "notes": notes,
        "pulls": 1000,
        "expected": 1383,
    },
]


class ByterII(Byter):
    def __init__(self, notes: str) -> None:
        super().__init__(notes)
        self.remove_cats_muzzles()
        self.lcm = lcm(*[len(w) for w in self.wheels])

    def total_byte_coins_after_lcm_version(self, pulls: int) -> int:
        if pulls <= self.lcm:
            return self.total_byte_coins_after(pulls)

        quotient, remainder = divmod(pulls, self.lcm)

        total = quotient * self.total_byte_coins_after(self.lcm)
        total += self.total_byte_coins_after(remainder)

        return total

    def total_byte_coins_after(self, pulls: int) -> int:

        byte_coins = 0

        # Precompute wheel positions for all pulls
        positions = [
            [
                self.wheels[i][t * self.deltas[i] % len(self.wheels[i])]
                for i in range(len(self.wheels))
            ]
            for t in range(1, pulls + 1)
        ]

        for pos in positions:
            b = sum(max(v - 2, 0) for v in Counter(chain.from_iterable(pos)).values())
            byte_coins += b

        return byte_coins

    def remove_cats_muzzles(self) -> None:
        for i in range(len(self.wheels)):
            for j in range(len(self.wheels[i])):
                self.wheels[i][j] = f"{self.wheels[i][j][0]}{self.wheels[i][j][-1]}"


@test(tests=tests[:])
def part_II(notes: str, pulls: int) -> int:
    byter = ByterII(notes)
    return byter.total_byte_coins_after_lcm_version(pulls)


Test Example Part II 202420242024 pulls passed, for part_II.
Test Example Part II 0 pull passed, for part_II.
Test Example Part II 1 time passed, for part_II.
Test Example Part II 2 pulls passed, for part_II.
Test Example Part II 3 pulls passed, for part_II.
Test Example Part II 4 pulls passed, for part_II.
Test Example Part II 5 pulls passed, for part_II.
Test Example Part II 10 pulls passed, for part_II.
Test Example Part II 100 pulls passed, for part_II.
Test Example Part II 1000 pulls passed, for part_II.
Success


In [16]:
with open("../inputs/everybody_codes_e2024_q16_p2.txt") as f:
    notes2 = f.read()

print(f"Part II: {part_II(notes2, 202420242024)}")

Part II: 141460607587


## Part III

Just as you expected, in the long run, it is impossible to win as much as you invest on any of the machines. Byter Paelish explains that anyone can check if they will win or not before pulling the right lever, and it is not his fault that people are simply lazy and prefer to rely on luck. However, the Knights of the Order are relentless and order the immediate shutdown of the machines.

But instead, Paelish instructs his employees to reset the wheels to their initial state, <span style="color: ivory">add a second lever to each machine on the left side and to hang an additional informational plaque</span> outlining the rules of the game. With his signature smile, he then asks the Knights of the Order to re-evaluate the machines. It appears that the old fox was prepared for a knightly inspection.

The informational plaque introduces the following rule: <span style="color: ivory">while all wheels have stopped, you may pull the lever on the left side of the machine to move all wheels downwards by one step,</span> or push it to move all wheels upwards by one step. You don't win any coins after this move alone, and you can do it only once or not at all before each pulling the right lever, which triggers the spin.

The inspectors decide that to verify the legality of the machines, knights need to calculate the <span style="color: ivory">maximum and minimum number of Byte Coins that can be won with <span style="color: ivory;border: 1px dashed lightyellow;width:fit-content; padding:5px;border-radius: 5px;">256</span> pulls of the right lever.</span>

Example based on the following notes:

<pre style="border: dashed green;width:fit-content;border-radius:5px;padding:2px;">
1,2,3

^_^ -.- ^,-
>.- ^_^ >.<
-_- -.- >.<
    -.^ ^_^
    >.>
</pre>

The machine starts by displaying the first trio of symbols: `^_^ -.- ^,-`.

Before pulling the right lever, you may push or pull the left one, changing the sequence to

- `>.- ^_^ >.<` - with a pull (the wheels move one step forward).
- `-_- >.> >.<` - with a push (the wheels move one step backward).

So, with the first pull of the right lever, the result might be:

<pre style="color: ivory;border: 1px solid MistyRose;width:fit-content; padding:5px;border-radius: 5px;">
Before the pull     After the pull      Byte Coins Won
 ^_^ -.- ^,-          >.- -.- >.<             1
 >.- ^_^ >.<          -_- -.^ ^,-             2
 -_- >.> >.<          ^_^ ^_^ ^.^             4 
</pre>

With a single right lever pull, <span style="color: ivory">the maximum number of coins you can win is 4 and the minimum number is 1,</span> so the answer would be `4 1`.

The results for further right lever pulls stand as follows:

- for 2 pulls: `6 1`
- for 3 pulls: `9 2`
- for 10 pulls: `26 5`
- for 100 pulls: `246 50`
- <span style="color: ivory">for 256 pulls: `627 128`</span>
- for 1000 pulls: `2446 500`
- for 2024 pulls: `4948 1012`

<span style="color: yellow">What is the maximum and the minimum number of Byte Coins that can be won on your new machine with 256 right lever pulls?</span>


In [ ]:
from functools import cache

from test_utilities import test

notes = """
1,2,3

^_^ -.- ^,-
>.- ^_^ >.<
-_- -.- ^.^
    -.^ >.<
    >.>
"""

tests = [
    {
        "name": "Example Part II 1 pull",
        "notes": notes,
        "pulls": 1,
        "expected": (4, 1),
    },
    {
        "name": "Example Part II 2 pulls",
        "notes": notes,
        "pulls": 2,
        "expected": (6, 1),
    },
    {
        "name": "Example Part II 3 pulls",
        "notes": notes,
        "pulls": 3,
        "expected": (9, 2),
    },
    {
        "name": "Example Part II 10 pulls",
        "notes": notes,
        "pulls": 10,
        "expected": (26, 5),
    },
    {
        "name": "Example Part II 100 pulls",
        "notes": notes,
        "pulls": 100,
        "expected": (246, 50),
    },
    {
        "name": "Example Part II 256 pulls",
        "notes": notes,
        "pulls": 256,
        "expected": (627, 128),
    },
    {
        "name": "Example Part II 1_000 pulls",
        "notes": notes,
        "pulls": 1_000,
        "expected": (2446, 500),
    },
    {
        "name": "Example Part II 2024 pulls",
        "notes": notes,
        "pulls": 2_024,
        "expected": (4948, 1012),
    },
]

INF = 10**18


class ByterIII(ByterII):
    def max_min_byte_coins(self, pulls: int) -> tuple[int, int]:
        n = len(self.wheels)

        # dp[(indici)] = (max_coins, min_coins)
        # Start with all possible initial positions (after 0 or ±1 adjustment)
        dp = {(0,) * n: (0, 0)}

        for pull in range(pulls):
            new_dp = {}
            for indici, (curr_max, curr_min) in dp.items():
                # Try all 3 lever positions: normal, +1, -1
                for delta in [0, 1, -1]:
                    next_indici = tuple(
                        (indici[i] + self.deltas[i] + delta) % len(self.wheels[i])
                        for i in range(n)
                    )
                    score = self.get_score(next_indici)
                    new_max = curr_max + score
                    new_min = curr_min + score

                    if next_indici not in new_dp:
                        new_dp[next_indici] = (new_max, new_min)
                    else:
                        old_max, old_min = new_dp[next_indici]
                        new_dp[next_indici] = (
                            max(old_max, new_max),
                            min(old_min, new_min),
                        )

            dp = new_dp

        # Find overall max and min
        max_coins = max(m for m, _ in dp.values())
        min_coins = min(m for _, m in dp.values())
        return max_coins, min_coins

    def max_min_byte_coins_dfs(self, pulls: int) -> tuple[int, int]:
        # gives stack overflow for the last test. Not for the answer of part III
        @cache
        def dfs(indici: tuple[int, ...], pulls) -> tuple[int, int]:
            if pulls == 0:
                return 0, 0

            min_bc, max_bc = INF, -INF

            for delta in [0, 1, -1]:

                indici1 = tuple(
                    (indici[i] + self.deltas[i] + delta) % len(self.wheels[i])
                    for i in range(n)
                )
                score = self.get_score(indici1)
                max_bc_0, min_bc_0 = dfs(indici1, pulls - 1)
                max_bc = max(max_bc, max_bc_0 + score)
                min_bc = min(min_bc, min_bc_0 + score)

            return max_bc, min_bc

        n = len(self.wheels)
        return dfs((0,) * n, pulls)

    @cache
    def get_score(self, indici: tuple[int, ...]) -> int:
        return sum(
            max(v - 2, 0)
            for v in Counter(
                chain.from_iterable(
                    self.wheels[i][indici[i]] for i in range(len(self.wheels))
                )
            ).values()
        )


@test(tests=tests[:])
def part_III(notes: str, pulls: int) -> tuple[int, int]:
    byter = ByterIII(notes)
    return byter.max_min_byte_coins(pulls)


Test Example Part II 1 pull passed, for part_III.
Test Example Part II 2 pulls passed, for part_III.
Test Example Part II 3 pulls passed, for part_III.
Test Example Part II 10 pulls passed, for part_III.
Test Example Part II 100 pulls passed, for part_III.
Test Example Part II 256 pulls passed, for part_III.
Test Example Part II 1_000 pulls passed, for part_III.
Test Example Part II 2024 pulls passed, for part_III.
Success


In [18]:
with open("../inputs/everybody_codes_e2024_q16_p3.txt") as f:
    notes2 = f.read()

print(f"Part II: {part_III(notes2, 256)}")

Part II: (611, 86)


![happy](./imgs/happy_quack.svg)
